# EDA on Jobtech API data

In [1]:
import requests
import json


url = 'https://jobsearch.api.jobtechdev.se'
url_for_search = f"{url}/search"


def _get_ads(params):
    headers = {'accept': 'application/json'}
    response = requests.get(url_for_search, headers=headers, params=params)
    response.raise_for_status()  # check for http errors
    return json.loads(response.content.decode('utf8'))

In [2]:
query = "python"
params = {'q': query, 'limit': 100}
json_response = _get_ads(params)
number_of_hits = json_response['total']['value']
print(f"\nNumber of hits = {number_of_hits}")


Number of hits = 743


In [9]:
json_response.keys()

dict_keys(['total', 'positions', 'query_time_in_millis', 'result_time_in_millis', 'stats', 'freetext_concepts', 'hits'])

In [11]:
json_response["hits"]

[{'relevance': 1.0,
  'id': '30751757',
  'external_id': None,
  'original_id': None,
  'label': [],
  'webpage_url': 'https://arbetsformedlingen.se/platsbanken/annonser/30751757',
  'logo_url': None,
  'headline': 'Fullstackutvecklare Python',
  'application_deadline': '2026-09-12T23:59:59',
  'number_of_vacancies': 1,
  'description': {'text': 'Har du funderat på att bli frilansare men ännu inte tagit steget? Hos oss får du kombinera tryggheten i en anställning med friheten att bygga din egen verksamhet.\nIndependtech är ett konsultbolag för IT-specialister med en ovanlig målsättning: när du säger upp dig från oss är det för att starta eget. Under tiden arbetar du i uppdrag som konsult, samtidigt som du får stöd, kunskap och ett nätverk som gör steget till frilanslivet enklare och tryggare.\n\nNu söker vi en erfaren fullstackutvecklare med fokus på Python:\nVem är du?\n\nDu har en\xa0dröm om att frilansa\xa0\n\n\nDu har 5+ års erfarenhet\xa0som utvecklare\n\n\nDu bor i Stockholmsområ

In [10]:
len(json_response["hits"])

100

In [6]:
hits = json_response["hits"]

for i, hit in enumerate(hits, start=1):
    print(i, f"{hit['headline']}, {hit['employer']['name']}")

1 Fullstackutvecklare Python, Independent Tech Sweden AB
2 Systemutvecklare med fokus Python, SVENSKA KRAFTNÄT
3 Python Backend Engineer (Transcripts), Quartr AB
4 Senior Data Engineer - Python, Techrytera AB
5 Senior backendutvecklare med python och nätverkskompetens, Combitech Aktiebolag
6 Systemutvecklare .NET / Java / Python, Liminity AB
7 Senior Software Engineer Python Azure DevOps, Avaron AB
8 Senior Python Developer to Husqvarna Group, Husqvarna AB
9 Python Backend Developer - AI & Data, Alten Sverige Aktiebolag
10 IT-säkerhetsspecialist, TERACOM AB
11 Bioinformatiker till enheten Exodiab, LUNDS UNIVERSITET
12 Är du en AI Platform Engineer och en Game Changer?, Barona Professionals AB
13 Accelerator Operator Assistant (part-time), LUNDS UNIVERSITET
14 Cybersäkerhetsanalytiker - Detection Engineering, Trafikverket
15 Systemtekniker inom Lagringsplattformar, Trafikverket
16 Spectrum Engineer, ACADEMIC WORK SWEDEN AB
17 ALTEN Linköping - Embedded C/C++ Developer, Alten Sverige Akt

# Understand pagination of Jobtech API
- pagination reduces the load on the server and the client by providing a subset of data at a time
- with limit-offset pagination, the client can specify the number of records to be retrieved (```limit```) and the starting point (```offset```) 

In [15]:
import pandas as pd

In [19]:
def fetch_jobs_offset(offset):
    all_jobs = []
    page_params = dict(params, offset=offset)
    data = _get_ads(page_params)

    for ad in data["hits"]:
        all_jobs.append(ad)

    df= pd.DataFrame(all_jobs)

    return df

In [22]:
df_offset0 = fetch_jobs_offset(0)
df_offset100 = fetch_jobs_offset(100)
print(f"Number of unique jobs fetched for offset 0: {df_offset0["id"].nunique()}")
print(f"Number of unique jobs fetched for offset 100: {df_offset100["id"].nunique()}")
print(f"The number of unique job ads from these two dataframes: {len(set(df_offset0["id"]) | set(df_offset100["id"]))}")



Number of unique jobs fetched for offset 0: 100
Number of unique jobs fetched for offset 100: 100
The number of unique job ads from these two dataframes: 200
